In [76]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


def add_derived_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # exact ratio used in the plot
    df["n_over_p"] = df["n"] / df["p"]

    # Optional: rounded ratio for printing/tables if you want (not used in grouping by default)
    df["n_over_p_round2"] = df["n_over_p"].round(2)

    # F1 (robust)
    pr = df["precision"].astype(float)
    rc = df["recall"].astype(float)
    denom = pr + rc
    df["f1"] = np.where(denom > 0, 2.0 * pr * rc / denom, np.nan)

    return df

def make_ebic_summary_table_by_p_and_np(
    df: pd.DataFrame,
    target_ratios=(1.5, 2.0),
    p_values=None,
    methods=("BIC", "EBIC"),
    decimals=3
) -> pd.DataFrame:
    """
    One row per (p, n/p, method). Useful for supplement.
    """
    df = add_derived_columns(df)
    df = df[df["method"].isin(methods)].copy()

    target_ratios = tuple(map(float, target_ratios))
    df = df[df["n_over_p"].isin(target_ratios)].copy()

    if p_values is not None:
        df = df[df["p"].isin(p_values)].copy()

    agg = df.groupby(["p", "n_over_p", "method"], as_index=False).agg(
        shd_mean=("shd", "mean"),
        shd_std=("shd", "std"),
        mse_mean=("mse", "mean"),
        mse_std=("mse", "std"),
        prec_mean=("precision", "mean"),
        prec_std=("precision", "std"),
        rec_mean=("recall", "mean"),
        rec_std=("recall", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        N=("shd", "count"),
    )

    def fmt(m, s):
        if pd.isna(m):
            return ""
        if pd.isna(s):
            return f"{m:.{decimals}f}"
        return f"{m:.{decimals}f} ± {s:.{decimals}f}"

    out = pd.DataFrame({
        "p": agg["p"],
        "n/p": agg["n_over_p"],
        "Method": agg["method"],
        "SHD": [fmt(m, s) for m, s in zip(agg["shd_mean"], agg["shd_std"])],
        "MSE": [fmt(m, s) for m, s in zip(agg["mse_mean"], agg["mse_std"])],
        "Precision": [fmt(m, s) for m, s in zip(agg["prec_mean"], agg["prec_std"])],
        "Recall": [fmt(m, s) for m, s in zip(agg["rec_mean"], agg["rec_std"])],
        "F1": [fmt(m, s) for m, s in zip(agg["f1_mean"], agg["f1_std"])],
        "N": agg["N"],
    })

    method_order = {m: i for i, m in enumerate(methods)}
    out = out.sort_values(by=["p", "n/p", "Method"],
                          key=lambda col: col.map(method_order) if col.name == "Method" else col)

    return out.reset_index(drop=True)


In [88]:
import numpy as np
import plotly.express as px
def plot_ebic_main_figure_plotly(
    df,
    target_ratios=(1.5, 2.0, 3.0, 4.0),
    tol=0.05,
    p_values=None,
    out_pdf="fig_ebic_main.pdf",
    out_png="fig_ebic_main.png",
    debug_points=False,
):

    df = df.copy()
    df["n_over_p"] = df["n"] / df["p"]
    df = df[df["method"].isin(["BIC", "EBIC"])].copy()

    # Keep only desired n/p regimes (within tolerance)
    keep = np.zeros(len(df), dtype=bool)
    for r in target_ratios:
        keep |= np.isclose(df["n_over_p"], r, atol=tol)
    df = df[keep].copy()

    # Snap to nearest target AND create a categorical label
    def snap_to_target(x):
        return float(min(target_ratios, key=lambda r: abs(x - r)))

    df["n_over_p"] = df["n_over_p"].apply(snap_to_target)

    # categorical x so Plotly doesn't drop ticks like 1.5
    df["np_bin"] = df["n_over_p"].map(lambda x: f"{x:.1f}")

    # Select p values to facet
    if p_values is None:
        p_values = sorted(df["p"].dropna().unique())[:3]
    df = df[df["p"].isin(p_values)].copy()

    fig = px.box(
        df,
        x="np_bin",                 # <-- categorical axis
        y="shd",
        color="method",
        facet_col="p",
        points="all" if debug_points else False,
        category_orders={
            "method": ["BIC", "EBIC"],
            "np_bin": [f"{r:.1f}" for r in target_ratios],  # includes 1.5
            "p": sorted(p_values),
        },
        labels={
            "np_bin": "Sample-to-variable ratio (n/p)",
            "shd": "SHD",
            "method": "",
            "p": "Number of variables (p)",
        },
        title="",
        color_discrete_map={"BIC": "#808080", "EBIC": "#009fd4"},
    )

    fig.update_layout(
        template="simple_white",
        legend_title_text="",
        margin=dict(l=60, r=30, t=30, b=55),
        font=dict(size=14),
    )

    # remove per-facet x title; keep only one global title
    fig.update_xaxes(title_text=None, showgrid=False)
    fig.update_yaxes(showgrid=True, gridcolor="rgba(0,0,0,0.08)")

    # facet title style: "p = 100"
    fig.for_each_annotation(lambda a: a.update(text=f"p = {a.text.split('=')[-1].strip()}"))

    # global x label (centered)
    fig.add_annotation(
        x=0.5, y=-0.13,
        xref="paper", yref="paper",
        text="Sample-to-variable ratio (n/p)",
        showarrow=False,
        font=dict(size=14),
    )

    # legend placement
    fig.update_layout(
        legend=dict(x=1.02, y=1.0, xanchor="left", yanchor="top", itemsizing="constant")
    )

    fig.write_image(out_pdf)
    fig.write_image(out_png, scale=3)
    return fig


In [84]:
df = pd.read_csv("./experiments/exp2/results.csv")
df['s']=df['s'].round(3)
plot_ebic_main_figure_plotly(df, target_ratios=(1.5, 2.0), p_values=[60, 90, 100], debug_points=False)

In [ ]:
# double checking entries in the dataset using the chart
df[(df['run'] == 15) & (df['n'] == 90) & (df['shd'] == 182) & (df['p'] == 60)]

,run,method,p,n,s,seed,mse,shd,precision,recall
2031,15,EBIC,60,90,0.068,15,0.023438,182,0.333333,0.669725


In [49]:
table = make_ebic_summary_table_by_p_and_np(df, target_ratios=(1.5,2.0))


In [50]:
table

,p,n/p,Method,SHD,MSE,Precision,Recall,F1,N
0,30,1.5,BIC,98.533 ± 39.857,0.059 ± 0.037,0.248 ± 0.081,0.550 ± 0.126,0.340 ± 0.098,300
1,30,1.5,EBIC,86.183 ± 40.419,0.059 ± 0.038,0.287 ± 0.103,0.534 ± 0.124,0.371 ± 0.113,300
2,30,2.0,BIC,83.287 ± 39.944,0.050 ± 0.035,0.318 ± 0.096,0.649 ± 0.123,0.425 ± 0.109,300
3,30,2.0,EBIC,70.647 ± 37.964,0.049 ± 0.035,0.372 ± 0.119,0.636 ± 0.125,0.466 ± 0.124,300
4,60,1.5,BIC,217.920 ± 101.317,0.025 ± 0.015,0.263 ± 0.076,0.662 ± 0.088,0.374 ± 0.089,300
5,60,1.5,EBIC,164.730 ± 86.658,0.025 ± 0.014,0.340 ± 0.106,0.646 ± 0.092,0.442 ± 0.109,300
6,60,2.0,BIC,162.333 ± 91.796,0.019 ± 0.014,0.369 ± 0.115,0.772 ± 0.088,0.495 ± 0.120,300
7,60,2.0,EBIC,124.440 ± 74.888,0.019 ± 0.013,0.446 ± 0.133,0.763 ± 0.094,0.558 ± 0.129,300
8,90,1.5,BIC,315.687 ± 161.745,0.015 ± 0.009,0.287 ± 0.077,0.725 ± 0.064,0.408 ± 0.086,300
9,90,1.5,EBIC,229.310 ± 122.767,0.015 ± 0.008,0.369 ± 0.095,0.712 ± 0.069,0.483 ± 0.096,300


In [51]:
table = table[['p', 'n/p', 'Method','SHD', 'MSE', 'Precision', 'Recall', 'F1']]

In [52]:
table[table['p'] == 100]

,p,n/p,Method,SHD,MSE,Precision,Recall,F1
12,100,1.5,BIC,365.840 ± 189.456,0.014 ± 0.008,0.283 ± 0.074,0.735 ± 0.061,0.405 ± 0.084
13,100,1.5,EBIC,262.160 ± 140.208,0.013 ± 0.007,0.366 ± 0.091,0.724 ± 0.065,0.483 ± 0.093
14,100,2.0,BIC,256.150 ± 170.806,0.010 ± 0.008,0.407 ± 0.125,0.835 ± 0.064,0.541 ± 0.125
15,100,2.0,EBIC,190.473 ± 129.999,0.010 ± 0.008,0.488 ± 0.135,0.828 ± 0.068,0.608 ± 0.125


In [54]:
df_double_check = df.copy()
df_double_check['n/p'] = df['n']/df['p']

In [75]:
methods_ = ["BIC", "EBIC"]
scores_ = ["mse", "precision", "recall", "shd"]
for m in methods_:
    for s in scores_:
        computed_mean = np.round(df_double_check[(df_double_check['n/p'] == 1.5) & (df_double_check['p'] == 100) & (df_double_check['method'] == m)][s].mean(),3)
        computed_std = np.round(df_double_check[(df_double_check['n/p'] == 1.5) & (df_double_check['p'] == 100) & (df_double_check['method'] == m)][s].std(),3)
        print(f"method: {m}, {s}: {computed_mean} +- {computed_std}")
        print("\n")

method: BIC, mse: 0.014 +- 0.008


method: BIC, precision: 0.283 +- 0.074


method: BIC, recall: 0.735 +- 0.061


method: BIC, shd: 365.84 +- 189.456


method: EBIC, mse: 0.013 +- 0.007


method: EBIC, precision: 0.366 +- 0.091


method: EBIC, recall: 0.724 +- 0.065


method: EBIC, shd: 262.16 +- 140.208




## Exp 3

In [89]:
df = pd.read_csv("./experiments/exp3/results.csv")
df['s']=df['s'].round(3)
plot_ebic_main_figure_plotly(df, target_ratios=(1.5, 2.0, 3.0, 4.0), p_values=[50, 100, 150], debug_points=False)

In [90]:
table = make_ebic_summary_table_by_p_and_np(df, p_values=[100], target_ratios=(1.5,2.0,3.0,4.0))


In [91]:
table

,p,n/p,Method,SHD,MSE,Precision,Recall,F1,N
0,100,1.5,BIC,365.840 ± 189.456,0.014 ± 0.008,0.283 ± 0.074,0.735 ± 0.061,0.405 ± 0.084,300
1,100,1.5,EBIC,262.160 ± 140.208,0.013 ± 0.007,0.366 ± 0.091,0.724 ± 0.065,0.483 ± 0.093,300
2,100,2.0,BIC,256.150 ± 170.806,0.010 ± 0.008,0.407 ± 0.125,0.835 ± 0.064,0.541 ± 0.125,300
3,100,2.0,EBIC,190.473 ± 129.999,0.010 ± 0.008,0.488 ± 0.135,0.828 ± 0.068,0.608 ± 0.125,300
4,100,3.0,BIC,78.483 ± 72.695,0.003 ± 0.003,0.709 ± 0.136,0.961 ± 0.037,0.811 ± 0.108,300
5,100,3.0,EBIC,57.507 ± 58.171,0.003 ± 0.003,0.774 ± 0.132,0.960 ± 0.038,0.853 ± 0.100,300
6,100,4.0,BIC,28.080 ± 32.206,0.001 ± 0.001,0.863 ± 0.086,0.993 ± 0.014,0.921 ± 0.059,300
7,100,4.0,EBIC,18.257 ± 25.710,0.001 ± 0.001,0.909 ± 0.078,0.993 ± 0.014,0.947 ± 0.052,300


## In-Degree Out-degree and Degree Distribution Charts

In [1]:
import os
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

def generate_comparative_network_distributions(
    datasets_info, 
    colors={"BIC": "#808080", "EBIC": "#009fd4"}, 
    output_folder="plots"
):
    """
    Reads nodes and edges CSVs for multiple datasets and exports three comparative 
    bar charts: Total Degree, In-Degree, and Out-Degree.
    
    Parameters:
    - datasets_info: dict mapping labels to paths, e.g., 
                     {"BIC": {"nodes": "path.csv", "edges": "path.csv"}}
    - colors: dict mapping dataset labels to hex color codes.
    - output_folder: string path for saving plots.
    """
    # 1. Build Directed Graphs for all datasets
    graphs = {}
    for label, paths in datasets_info.items():
        nodes_df = pd.read_csv(paths['nodes'])
        edges_df = pd.read_csv(paths['edges'])
        
        G = nx.from_pandas_edgelist(
            edges_df, source='Source', target='Target', create_using=nx.DiGraph()
        )
        G.add_nodes_from(nodes_df['Id'])
        graphs[label] = G

    # 2. Configure Publication Styling (Arial)
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Arial']
    
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # 3. Define metrics to calculate
    metrics = ["Total_Degree", "In_Degree", "Out_Degree"]

    # 4. Generate and Save the Comparative Charts
    for metric in metrics:
        all_counts = {}
        
        # Calculate degrees for each dataset
        for label, G in graphs.items():
            if metric == "Total_Degree":
                deg_dict = dict(G.degree())
            elif metric == "In_Degree":
                deg_dict = dict(G.in_degree())
            else: # Out_Degree
                deg_dict = dict(G.out_degree())
            
            degrees = list(deg_dict.values())
            # Count frequencies
            all_counts[label] = pd.Series(degrees).value_counts()
            
        # Combine counts into a single DataFrame to align the indexes (degree values)
        # Any degree present in one dataset but missing in another is filled with 0
        df_counts = pd.DataFrame(all_counts).fillna(0).sort_index()
        
        # Set up plot
        plt.figure(figsize=(8, 5))
        ax = plt.gca()
        
        labels = list(datasets_info.keys())
        n_datasets = len(labels)
        
        # Grouped bar positioning math
        total_width = 0.8
        bar_width = total_width / n_datasets
        
        for i, label in enumerate(labels):
            # Calculate the x offset so bars sit side-by-side
            offset = (i - n_datasets / 2) * bar_width + bar_width / 2
            
            # Use the actual degree value (df_counts.index) as the base x-coordinate
            x_positions = df_counts.index.values + offset
            
            # Use the requested color mapping, fallback to black if missing
            bar_color = colors.get(label, '#000000') 
            
            ax.bar(
                x_positions, 
                df_counts[label], 
                width=bar_width, 
                label=label, 
                color=bar_color, 
                edgecolor='white', 
                linewidth=0.5
            )

        # Labels & Titles
        display_name = metric.replace("_", " ")
        plt.title(f"{display_name} Comparison", fontsize=15, fontweight='bold', pad=15)
        plt.xlabel("Degree Value", fontsize=12)
        plt.ylabel("Frequency (Node Count)", fontsize=12)
        
        # Ensure x-axis ticks are integers (since degrees can't be fractions)
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

        # Visual Clean-up
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.grid(axis='y', linestyle='--', alpha=0.4)
        
        # Add Legend
        plt.legend(frameon=False, fontsize=11)
        
        # Save high-resolution PNG
        filename = f"comparison_{metric.lower()}.png"
        plt.savefig(os.path.join(output_folder, filename), dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"Generated: {filename}")

In [2]:
my_datasets = {
    "BIC": {
        "nodes": "./real_data_application/exp4/output/lingam/nodes.csv",
        "edges": "./real_data_application/exp4/output/lingam/edges.csv"
    },
    "EBIC": {
        "nodes": "./real_data_application/exp4/output/eBIC/nodes.csv",
        "edges": "./real_data_application/exp4/output/eBIC/edges.csv"
    }
}

generate_comparative_network_distributions(datasets_info=my_datasets, output_folder="./")

Generated: comparison_total_degree.png
Generated: comparison_in_degree.png
Generated: comparison_out_degree.png
